# Lab 8


## Setup for SUSY Dataset

Use the SUSY dataset for the rest of this lab. Here is a basic setup.

In [ ]:
# Our usual libraries...
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import HTML, display
import tabulate

In [ ]:
filename="../Lab.7/SUSY.csv"
VarNames=["signal", "l_1_pT", "l_1_eta","l_1_phi", "l_2_pT", "l_2_eta", 
          "l_2_phi", "MET", "MET_phi", "MET_rel", "axial_MET",
          "M_R", "M_TR_2", "R", "MT2", "S_R", "M_Delta_R", "dPhi_r_b", "cos_theta_r1"]
df = pd.read_csv(filename, dtype='float64', names=VarNames)

## Scikit-Learn

[Scikit-learn](http://scikit-learn.org) is a rich python library for data science, including machine learning. For example, we can build a Fisher Discriminant (aka Linear Discriminant Analysis, or LDA). 

### Exercise 1: Install Scikit-Learn

Follow the [Installation Instructions](https://scikit-learn.org/stable/install.html) and install `scikit-learn` in your environment.

### Exercise 2: Read About Classifiers

#### Part a
Scikit-learn offers an impressively comprehensive list of machine learning algorithms. Browse through [scikit-learn's documentation](https://scikit-learn.org/stable/index.html). You'll note the algorithms are organized into classification, regression, clustering, dimensionality reduction, model selection, and preprocessing. Browse through the list of [classification algorithms](https://scikit-learn.org/stable/supervised_learning.html#supervised-learning). 

#### Part b
Note scikit-learn's documentation is rather comprehensive. The documentation on [linear models](https://scikit-learn.org/stable/modules/linear_model.html) shows how classification problems are setup. Read about the first few methods and try to comprehend the example codes. Skim the rest of the document.

#### Part c
Read through the [LDA Documentation](https://scikit-learn.org/stable/modules/lda_qda.html).


### Exercise 3: Training a Classifier

Lets' repeat what we did manually in the previous lab using scikit-learn. We'll use a LDA classifier, which we can instanciate as follows:

In [ ]:
import sklearn.discriminant_analysis as DA
Fisher=DA.LinearDiscriminantAnalysis()

As discussed in the lecture, to properly formulate our problem, we'll have to:

* Define the inputs (X) vs outputs (Y)
* Designate training vs testing samples (in order to get a unbias assessment of the performance of Machine Learning algorithms)

for example, here we'll take use 4M events for training and the remainder for testing.

In [ ]:
N_Train=4000000

Train_Sample=df[:N_Train]
Test_Sample=df[N_Train:]

X_Train=Train_Sample[VarNames[1:]]
y_Train=Train_Sample["signal"]

X_Test=Test_Sample[VarNames[1:]]
y_Test=Test_Sample["signal"]

Test_sig=Test_Sample[Test_Sample.signal==1]
Test_bkg=Test_Sample[Test_Sample.signal==0]

##add training variables for code below
Train_sig = Train_Sample[Train_Sample.signal == 1]
Train_bkg = Train_Sample[Train_Sample.signal == 0]

We can train the classifier as follow:

In [ ]:
Fisher.fit(X_Train,y_Train)

We can plot the output, comparing signal and background:

In [ ]:
plt.figure()
plt.hist(Fisher.decision_function(Test_sig[VarNames[1:]]),bins=100,histtype="step", color="blue", label="signal",stacked=True)
plt.hist(Fisher.decision_function(Test_bkg[VarNames[1:]]),bins=100,histtype="step", color="red", label="background",stacked=True)
plt.legend(loc='upper right')
plt.show()

#### Part a

Compare ROC curves computed on the test versus training samples, in a single plot. Do you see a bias?

In [ ]:
def compute_roc_from_scores(scores_sig, scores_bkg, n_cuts=500):

    ##gathers scores of signals
    all_scores = np.concatenate([scores_sig, scores_bkg])
    xc_values  = np.linspace(all_scores.min(), all_scores.max(), n_cuts)

    ##creates lists
    tpr_list = []
    fpr_list = []

    for xc in xc_values:
        tpr_list.append(np.mean(scores_sig > xc))
        fpr_list.append(np.mean(scores_bkg > xc))

    fpr_arr  = np.array(fpr_list)
    tpr_arr  = np.array(tpr_list)

    sort_idx = np.argsort(fpr_arr)
    fpr_arr  = fpr_arr[sort_idx]
    tpr_arr  = tpr_arr[sort_idx]
    auc_val  = auc(fpr_arr, tpr_arr)

    return fpr_arr, tpr_arr, auc_val


##Computes the scores for both the test and training samples
test_scores_sig  = Fisher.decision_function(Test_sig[VarNames[1:]])
test_scores_bkg  = Fisher.decision_function(Test_bkg[VarNames[1:]])
train_scores_sig = Fisher.decision_function(Train_sig[VarNames[1:]])
train_scores_bkg = Fisher.decision_function(Train_bkg[VarNames[1:]])

##Computes the ROC curves
fpr_test,  tpr_test,  auc_test  = compute_roc_from_scores(test_scores_sig, test_scores_bkg)
fpr_train, tpr_train, auc_train = compute_roc_from_scores(train_scores_sig, train_scores_bkg)

##Plots them
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(fpr_train, tpr_train, color='blue', linewidth=2, label=f'Training Sample (AUC={auc_train:.4f})')
ax.plot(fpr_test,  tpr_test,  color='red',  linewidth=2, label=f'Test Sample (AUC={auc_test:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')

ax.set_xlabel('False Positive Rate (Background Efficiency)', fontsize=12)
ax.set_ylabel('True Positive Rate (Signal Efficiency)',      fontsize=12)
ax.set_title('ROC Curves: Test vs Training Sample\n(Fisher LDA)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f"Training AUC : {auc_train:.6f}")
print(f"Test AUC     : {auc_test:.6f}")
print(f"AUC Difference (bias indicator): {abs(auc_train - auc_test):.6f}")

In [ ]:
As seen in the AUC difference, there is almost no bias shown between the training and test
samples. The Fisher LDA does not have parameters to tune, so the ROC curves are extremely
similar.

#### Part b

Train the Fisher performance of using the raw, features, and raw+features as input. Compare the performance one a single plot. 

In [ ]:
##Feature groups
low_level  = ["l_1_pT", "l_1_eta", "l_1_phi",
              "l_2_pT", "l_2_eta", "l_2_phi",
              "MET", "MET_phi"]

high_level = ["MET_rel", "axial_MET", "M_R", "M_TR_2",
              "R", "MT2", "S_R", "M_Delta_R",
              "dPhi_r_b", "cos_theta_r1"]

all_features = low_level + high_level

feature_sets = {
    "Raw (Low-Level) Features"      : low_level,
    "High-Level Features"           : high_level,
    "Raw + High-Level (All)"        : all_features,
}

colors = ['green', 'orange', 'red']

fig, ax = plt.subplots(figsize=(9, 7))

for (label, features), color in zip(feature_sets.items(), colors):
    ##Trains a fisher on this feature table
    fisher = DA.LinearDiscriminantAnalysis()
    fisher.fit(Train_Sample[features], y_Train)

    ##Score on the test sample
    scores_sig = fisher.decision_function(Test_sig[features])
    scores_bkg = fisher.decision_function(Test_bkg[features])

    fpr, tpr, auc_val = compute_roc_from_scores(scores_sig, scores_bkg)

    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{label} (AUC={auc_val:.4f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')

ax.set_xlabel('False Positive Rate (Background Efficiency)', fontsize=12)
ax.set_ylabel('True Positive Rate (Signal Efficiency)',      fontsize=12)
ax.set_title('Fisher LDA: ROC Curves by Feature Set',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

### Exercise 4: Comparing Techniques

#### Part a
Select 3 different classifiers from the techniques listed [here](http://scikit-learn.org/stable/supervised_learning.html#supervised-learning) to compare. Note that you can use the multi-layer perceptron to build a deep network, though training may be prohibitively slow. So avoid this technique.

#### Part b

Write a function that takes an instantiated classifier and performs the comparison from part 3b. Use the function on your choice of functions in part a.

#### Part c

Use the best method from part c to compute the maximal significance $\sigma_S= \frac{N_S}{\sqrt{N_S+N_B}}$ for the scenarios in lab 5.

In [ ]:
##Part A

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
import sklearn.discriminant_analysis as DA

classifiers = {
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42,
        n_jobs=-1
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ),
}

In [ ]:
##Part B

def compare_classifier(clf, clf_name, X_Train, y_Train, Test_sig, Test_bkg, Train_sig, Train_bkg,
                        features=None, n_cuts=300, ax=None, color='blue'):
##parameters as listed
        ##clf       : instantiated sklearn classifier
        ##clf_name  : string def for the classifier
        ##X_Train   : training features for DataFrame
        ##y_Train   : training name for Series
        ##Test_sig  : tests signal DataFrame
        ##Test_bkg  : test background DataFrame
        ##Train_sig : training signal DataFrame
        ##Train_bkg : training background DataFrame
        ##features  : list of feature groups
        ##n_cuts    : number of cuts for ROC curves
        ##ax        : matplotlib axis to plot on
        ##color     : color for this classifier's curves

    if features is None:
        features = VarNames[1:]

    ##Trains
    print(f"Training {clf_name}...")
    clf.fit(X_Train[features], y_Train)
    print(f"Done.")

    ##Gets scores by using predict_proba.If not, uses decision_function instead
    def get_scores(clf, X):
        if hasattr(clf, 'predict_proba'):
            return clf.predict_proba(X)[:, 1]
        else:
            return clf.decision_function(X)

    test_scores_sig  = get_scores(clf, Test_sig[features])
    test_scores_bkg  = get_scores(clf, Test_bkg[features])
    train_scores_sig = get_scores(clf, Train_sig[features])
    train_scores_bkg = get_scores(clf, Train_bkg[features])

    fpr_test,  tpr_test,  auc_test  = compute_roc_from_scores(
        test_scores_sig,  test_scores_bkg,  n_cuts=n_cuts)
    fpr_train, tpr_train, auc_train = compute_roc_from_scores(
        train_scores_sig, train_scores_bkg, n_cuts=n_cuts)

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))

    ax.plot(fpr_test,  tpr_test,  color=color, linewidth=2, label=f'{clf_name} Test  (AUC={auc_test:.4f})')
    ax.plot(fpr_train, tpr_train, color=color, linewidth=2, linestyle='--', label=f'{clf_name} Train (AUC={auc_train:.4f})')

    return auc_train, auc_test


##Then compare all 3 classifiers with using Fisher as a baseline
fig, ax = plt.subplots(figsize=(10, 7))

clf_colors = ['red', 'green', 'orange']
auc_results = {}

for (clf_name, clf), color in zip(classifiers.items(), clf_colors):
    auc_train, auc_test = compare_classifier(
        clf, clf_name,
        X_Train, y_Train,
        Test_sig, Test_bkg,
        Train_sig, Train_bkg,
        ax=ax, color=color
    )
    auc_results[clf_name] = {'train': auc_train, 'test': auc_test}

##Adds Fisher as baseline
fisher = DA.LinearDiscriminantAnalysis()
fisher.fit(X_Train, y_Train)
fpr_f, tpr_f, auc_f = compute_roc_from_scores(
    fisher.decision_function(Test_sig[VarNames[1:]]),
    fisher.decision_function(Test_bkg[VarNames[1:]])
)
ax.plot(fpr_f, tpr_f, color='blue', linewidth=2, label=f'Fisher LDA Baseline (AUC={auc_f:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')

ax.set_xlabel('False Positive Rate (Background Efficiency)', fontsize=12)
ax.set_ylabel('True Positive Rate (Signal Efficiency)',      fontsize=12)
ax.set_title('ROC Curves: Classifier Comparison\n(dashed=train, solid=test)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

##AUC summary table
print(f"\n{'Classifier':<25} {'Train AUC':>10} {'Test AUC':>10} {'Overfit Gap':>12}")
print("-" * 60)
for name, aucs in auc_results.items():
    gap = aucs['train'] - aucs['test']
    print(f"{name:<25} {aucs['train']:>10.4f} {aucs['test']:>10.4f} {gap:>12.4f}")

In [ ]:
##Lab 5 scenarios
scenarios = [
    {'Ns': 10,    'Nb': 100},
    {'Ns': 100,   'Nb': 1000},
    {'Ns': 1000,  'Nb': 10000},
    {'Ns': 10000, 'Nb': 100000},
]

scenario_colors = ['purple', 'green', 'orange', 'brown']
scenario_labels = [f"$N_s$={s['Ns']}, $N_b$={s['Nb']}" for s in scenarios]

def max_significance_classifier(clf, clf_name, Test_sig, Test_bkg, scenarios, features=None,
n_cuts=500, figsize=(9, 5)):
    if features is None:
        features = VarNames[1:]

    def get_scores(clf, X):
        if hasattr(clf, 'predict_proba'):
            return clf.predict_proba(X)[:, 1]
        else:
            return clf.decision_function(X)

    scores_sig = get_scores(clf, Test_sig[features])
    scores_bkg = get_scores(clf, Test_bkg[features])

    all_scores = np.concatenate([scores_sig, scores_bkg])
    xc_values  = np.linspace(all_scores.min(), all_scores.max(), n_cuts)

    fig, ax = plt.subplots(figsize=figsize)

    print(f"\nMaximal Significance — {clf_name}")
    print(f"{'Scenario':<30} {'Max σ':>10} {'Optimal xc':>12}")
    print("-" * 55)

    for scenario, color, label in zip(scenarios, scenario_colors, scenario_labels):
        Ns = scenario['Ns']
        Nb = scenario['Nb']

        significance = []
        for xc in xc_values:
            es    = np.mean(scores_sig > xc)
            eb    = np.mean(scores_bkg > xc)
            Ns_p  = es * Ns
            Nb_p  = eb * Nb
            denom = np.sqrt(Ns_p + Nb_p)
            sig   = Ns_p / denom if denom > 0 else 0
            significance.append(sig)

        significance = np.array(significance)
        max_sig      = np.nanmax(significance)
        best_xc      = xc_values[np.nanargmax(significance)]

        ax.plot(xc_values, significance, color=color,
                linewidth=1.5, label=f'{label} (max σ={max_sig:.3f})')
        ax.axvline(best_xc, color=color, linewidth=0.8, linestyle=':')

        print(f"{label:<30} {max_sig:>10.4f} {best_xc:>12.4f}")

    ax.set_xlabel('Classifier Score Threshold $x_c$', fontsize=12)
    ax.set_ylabel(r'Significance $\sigma$',           fontsize=12)
    ax.set_title(f'Max Significance: {clf_name}',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()


##Looks for the best classifier by test AUC
best_name = max(auc_results, key=lambda k: auc_results[k]['test'])
best_clf  = classifiers[best_name]
print(f"Best classifier: {best_name} (Test AUC={auc_results[best_name]['test']:.4f})")

##Computes the max significance with best classifier
max_significance_classifier(best_clf, best_name, Test_sig, Test_bkg, scenarios)

### Exercise 5: Metrics

Scikit-learn provides methods for computing the FPR, TPR, ROC, AUC metrics. For example:

In [ ]:
from sklearn.metrics import roc_curve, auc
fpr, tpr, _ = roc_curve(y_Test, Fisher.decision_function(X_Test))

roc_auc = auc(fpr, tpr)

plt.plot(fpr,tpr,color='darkorange',label='ROC curve (area = %0.2f)' % roc_auc)
plt.legend(loc="lower right")
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')

plt.show()


#### Part a
TPR/FPR/ROC/AUC are one way of assessing the quality of a classifier. Read about [Precision and Recall](https://en.wikipedia.org/wiki/Precision_and_recall), [Accuracy](https://en.wikipedia.org/wiki/Accuracy_and_precision), and [F-score](https://en.wikipedia.org/wiki/F-score).

#### Part b
Look through [model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html#) documentation. Using scikit-learns tools, compute TPR, FPR, ROC, AUC, Precision, Recall, F1 score, and accuracy for the method you selected in 4c above and each scenario. Make a nice table, which also includes the maximal significance. 


In [ ]:
##Part B

from sklearn.metrics import (roc_curve, auc, precision_score, recall_score, f1_score, accuracy_score, confusion_matrix)
from IPython.display import HTML, display
import tabulate as tb
import numpy as np

def compute_all_metrics(clf, clf_name, X_Test, y_Test, Test_sig, Test_bkg, scenarios, features=None, n_cuts=500):
    if features is None:
        features = VarNames[1:]

    ##Get scores and predictions
    def get_scores(clf, X):
        if hasattr(clf, 'predict_proba'):
            return clf.predict_proba(X)[:, 1]
        else:
            return clf.decision_function(X)

    scores      = get_scores(clf, X_Test[features])
    scores_sig  = get_scores(clf, Test_sig[features])
    scores_bkg  = get_scores(clf, Test_bkg[features])
    y_pred      = clf.predict(X_Test[features])

    ##Sklearn ROC and AUC
    fpr_arr, tpr_arr, thresholds = roc_curve(y_Test, scores)
    roc_auc = auc(fpr_arr, tpr_arr)

    ##Plots ROC curve
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(fpr_arr, tpr_arr, color='darkorange', linewidth=2, label=f'ROC (AUC={roc_auc:.4f})')
    axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
    axes[0].set_xlabel('False Positive Rate', fontsize=12)
    axes[0].set_ylabel('True Positive Rate',  fontsize=12)
    axes[0].set_title(f'ROC Curve: {clf_name}', fontsize=13, fontweight='bold')
    axes[0].legend(fontsize=10, loc='lower right')
    axes[0].spines['top'].set_visible(False)
    axes[0].spines['right'].set_visible(False)

    ##Compares Significance vs threshold for all scenarios
    all_scores = np.concatenate([scores_sig, scores_bkg])
    xc_values  = np.linspace(all_scores.min(), all_scores.max(), n_cuts)

    scenario_colors = ['purple', 'green', 'orange', 'brown']
    scenario_labels = [f"Ns={s['Ns']}, Nb={s['Nb']}" for s in scenarios]

    max_sigs   = {}
    best_xcs   = {}
    best_tprs  = {}
    best_fprs  = {}

    for scenario, color, label in zip(scenarios, scenario_colors, scenario_labels):
        Ns = scenario['Ns']
        Nb = scenario['Nb']
        significance = []

        for xc in xc_values:
            es    = np.mean(scores_sig > xc)
            eb    = np.mean(scores_bkg > xc)
            Ns_p  = es * Ns
            Nb_p  = eb * Nb
            denom = np.sqrt(Ns_p + Nb_p)
            sig   = Ns_p / denom if denom > 0 else 0
            significance.append(sig)

        significance  = np.array(significance)
        best_idx      = np.nanargmax(significance)
        max_sig       = significance[best_idx]
        best_xc       = xc_values[best_idx]

        max_sigs[label]  = max_sig
        best_xcs[label]  = best_xc
        best_tprs[label] = np.mean(scores_sig > best_xc)
        best_fprs[label] = np.mean(scores_bkg > best_xc)

        axes[1].plot(xc_values, significance, color=color, linewidth=1.5, label=f'{label} (max σ={max_sig:.3f})')
        axes[1].axvline(best_xc, color=color, linewidth=0.8, linestyle=':')

    axes[1].set_xlabel('Classifier Score Threshold', fontsize=12)
    axes[1].set_ylabel(r'Significance $\sigma$',     fontsize=12)
    axes[1].set_title('Significance vs Threshold',   fontsize=13, fontweight='bold')
    axes[1].legend(fontsize=8)
    axes[1].spines['top'].set_visible(False)
    axes[1].spines['right'].set_visible(False)

    plt.suptitle(f'Metrics Overview: {clf_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    ##Global metrics, like for threshold-independent or at default
    precision = precision_score(y_Test, y_pred)
    recall    = recall_score(y_Test, y_pred)
    f1        = f1_score(y_Test, y_pred)
    accuracy  = accuracy_score(y_Test, y_pred)

    tn, fp, fn, tp = confusion_matrix(y_Test, y_pred).ravel()
    global_tpr = tp / (tp + fn)
    global_fpr = fp / (fp + tn)

    ##Constructs metrics table
    ##Row 1 for global metrics
    rows = []

    ##Header row for global metrics
    global_row = [
        "Global (default threshold)",
        f"{global_tpr:.4f}",
        f"{global_fpr:.4f}",
        f"{roc_auc:.4f}",
        f"{precision:.4f}",
        f"{recall:.4f}",
        f"{f1:.4f}",
        f"{accuracy:.4f}",
        "—"
    ]
    rows.append(global_row)

    ##One row for each scenario at optimal threshold
    for scenario, label in zip(scenarios, scenario_labels):
        Ns  = scenario['Ns']
        Nb  = scenario['Nb']
        xc  = best_xcs[label]
        tpr = best_tprs[label]
        fpr = best_fprs[label]

        ##Recomputes precision, recall, and f1 qualities at optimal threshold
        y_pred_opt = (scores > xc).astype(int)
        prec_opt   = precision_score(y_Test, y_pred_opt, zero_division=0)
        rec_opt    = recall_score(y_Test, y_pred_opt,    zero_division=0)
        f1_opt     = f1_score(y_Test, y_pred_opt,        zero_division=0)
        acc_opt    = accuracy_score(y_Test, y_pred_opt)

        rows.append([
            f"Optimal xc ({label})",
            f"{tpr:.4f}",
            f"{fpr:.4f}",
            f"{roc_auc:.4f}",
            f"{prec_opt:.4f}",
            f"{rec_opt:.4f}",
            f"{f1_opt:.4f}",
            f"{acc_opt:.4f}",
            f"{max_sigs[label]:.4f}",
        ])

    headers = ["Threshold", "TPR", "FPR", "AUC", "Precision", "Recall", "F1", "Accuracy", "Max σ"]

    html = tb.tabulate(rows, tablefmt='html', headers=headers)
    styled = f"""
    <h3 style='font-family:sans-serif; color:#333'>
        Metrics Table: {clf_name}
    </h3>
    <style>
        table {{border-collapse: collapse; font-family: monospace;
                font-size: 12px;}}
        th, td {{border: 1px solid #ccc; padding: 6px 12px;
                 text-align: right;}}
        th {{background-color: #f0f0f0; font-weight: bold;}}
        tr:nth-child(even) {{background-color: #f9f9f9;}}
        td:first-child, th:first-child {{text-align: left;}}
    </style>
    {html}
    """
    display(HTML(styled))

    return max_sigs


##Runs on best classifier from previous exercise
max_sigs = compute_all_metrics(
    best_clf, best_name,
    X_Test, y_Test,
    Test_sig, Test_bkg,
    scenarios
)